In [101]:
import pandas as pd
import numpy as np

users = pd.read_csv("users.csv")
experiments = pd.read_csv("experiments.csv")
events = pd.read_csv("events.csv")
transactions = pd.read_csv("transactions.csv")

In [102]:
users_clean = users.copy()
experiments_clean = experiments.copy()
events_clean = events.copy()
transactions_clean = transactions.copy()

In [103]:
users_clean['signup_date'] = pd.to_datetime(
    users_clean['signup_date'],
    errors='coerce'
)

In [104]:
users_clean['signup_date'].isna().sum()

0

In [105]:
users_clean = users_clean.drop_duplicates()

In [106]:
users_clean.shape

(100000, 6)

In [107]:
users_clean['user_id'].nunique()

100000

In [108]:
users_clean['country']=users_clean['country'].fillna('Unknown')

In [109]:
users_clean['country'].isna().sum()

0

In [110]:
users_clean['country'].value_counts()

country
India        34874
USA          24972
UK           11897
Canada        9995
Germany       9951
Australia     8111
Unknown        200
Name: count, dtype: int64

In [111]:
users_clean['device_type'].value_counts(dropna=False)

device_type
Mobile     61794
Desktop    29994
Tablet      8112
NaN          100
Name: count, dtype: int64

In [112]:
users_clean.isnull().sum()

user_id                  0
signup_date              0
country                  0
device_type            100
age_group                0
acquisition_channel      0
dtype: int64

In [113]:
users_clean.duplicated().sum()

0

In [114]:
users_clean['user_id'].nunique()

100000

In [115]:
events_clean.head()

,event_id,user_id,event_name,event_timestamp,session_id,platform,event_hour
0,E00000001,U000001,signup,2026-02-21 19:21:00,SU00000101,Desktop,19
1,E00000002,U000001,onboarding_start,2026-02-22 02:25:00,SU00000101,Desktop,2
2,E00000003,U000001,onboarding_complete,2026-02-23 21:22:00,SU00000101,Desktop,21
3,E00000004,U000001,core_action,2026-02-28 00:29:00,SU00000101,Desktop,0
4,E00000005,U000001,feature_view,2026-02-23 07:26:00,SU00000101,Desktop,7


In [116]:
events_clean.shape

(432790, 7)

In [117]:
events_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432790 entries, 0 to 432789
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   event_id         432790 non-null  object
 1   user_id          432790 non-null  object
 2   event_name       432790 non-null  object
 3   event_timestamp  432790 non-null  object
 4   session_id       432790 non-null  object
 5   platform         432790 non-null  object
 6   event_hour       432790 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 23.1+ MB


In [118]:
events_clean.isnull().sum()

event_id           0
user_id            0
event_name         0
event_timestamp    0
session_id         0
platform           0
event_hour         0
dtype: int64

In [119]:
events_clean.duplicated().sum()

250

In [120]:
events_clean['event_id'].duplicated().sum()

250

In [121]:
events_clean['user_id'].nunique()

100000

In [122]:
valid_user_ids = set(users_clean['user_id'])

orphan_events = events_clean[
    ~events_clean['user_id'].isin(valid_user_ids)
]

len(orphan_events)

0

In [123]:
events_clean['event_name'].value_counts(dropna=False)

event_name
signup                 100062
onboarding_start       100039
onboarding_complete     74930
core_action             73628
session_start           38959
feature_view            36810
subscription_start       8362
Name: count, dtype: int64

In [124]:
events_clean['platform'].value_counts(dropna=False)

platform
Mobile     267936
Desktop    129620
Tablet      35234
Name: count, dtype: int64

In [125]:
events_clean['event_timestamp'] = pd.to_datetime(
    events_clean['event_timestamp'],
    errors='coerce'
)

In [126]:
events_clean['event_timestamp'].isna().sum()

0

In [127]:
events_clean['event_timestamp'].min()

Timestamp('2026-01-01 00:00:00')

In [128]:
events_clean['event_timestamp'].max()

Timestamp('2026-04-30 23:40:00')

In [129]:
events_check = events_clean.merge(
    users_clean[['user_id', 'signup_date']],
    on='user_id',
    how='left'
)

In [130]:
events_before_signup = events_check[
    events_check['event_timestamp'].dt.date <
    events_check['signup_date'].dt.date
]

In [131]:
len(events_before_signup)

0

In [132]:
events_check = events_check.merge(
    experiments_clean[['user_id', 'assignment_date']],
    on='user_id',
    how='left'
)

In [133]:
events_before_assignment = events_check[
    events_check['event_timestamp'] <
    events_check['assignment_date']
]

In [134]:
len(events_before_assignment)

3047

In [135]:
events_before_assignment['event_name'].value_counts()

event_name
signup              2039
onboarding_start    1008
Name: count, dtype: int64

In [136]:
onboarding_events = events_clean[
    events_clean['event_name'].isin([
        'signup',
        'onboarding_start',
        'onboarding_complete',
        'core_action'
    ])
].sort_values(['user_id', 'event_timestamp'])

In [137]:
onboarding_events.head(30)

,event_id,user_id,event_name,event_timestamp,session_id,platform,event_hour
0,E00000001,U000001,signup,2026-02-21 19:21:00,SU00000101,Desktop,19
1,E00000002,U000001,onboarding_start,2026-02-22 02:25:00,SU00000101,Desktop,2
2,E00000003,U000001,onboarding_complete,2026-02-23 21:22:00,SU00000101,Desktop,21
5,E00000006,U000001,core_action,2026-02-26 23:32:00,SU00000101,Desktop,23
3,E00000004,U000001,core_action,2026-02-28 00:29:00,SU00000101,Desktop,0
9,E00000010,U000002,signup,2026-01-15 05:44:00,SU00000201,Mobile,5
10,E00000011,U000002,onboarding_start,2026-01-16 05:55:00,SU00000201,Mobile,5
11,E00000012,U000002,onboarding_complete,2026-01-17 12:11:00,SU00000201,Mobile,12
12,E00000013,U000003,signup,2026-03-13 02:08:00,SU00000301,Mobile,2
13,E00000014,U000003,onboarding_start,2026-03-13 23:29:00,SU00000301,Mobile,23


In [138]:
events_clean = events_clean.drop_duplicates()

In [139]:
print("Shape:", events_clean.shape)
print("Duplicate rows:", events_clean.duplicated().sum())
print("Duplicate event IDs:", events_clean['event_id'].duplicated().sum())

Shape: (432540, 7)
Duplicate rows: 0
Duplicate event IDs: 0


In [140]:
onboarding_events = events_clean[
    events_clean['event_name'].isin([
        'signup',
        'onboarding_start',
        'onboarding_complete',
        'core_action'
    ])
].sort_values(['user_id', 'event_timestamp'])

onboarding_events.head(10)

,event_id,user_id,event_name,event_timestamp,session_id,platform,event_hour
0,E00000001,U000001,signup,2026-02-21 19:21:00,SU00000101,Desktop,19
1,E00000002,U000001,onboarding_start,2026-02-22 02:25:00,SU00000101,Desktop,2
2,E00000003,U000001,onboarding_complete,2026-02-23 21:22:00,SU00000101,Desktop,21
5,E00000006,U000001,core_action,2026-02-26 23:32:00,SU00000101,Desktop,23
3,E00000004,U000001,core_action,2026-02-28 00:29:00,SU00000101,Desktop,0
9,E00000010,U000002,signup,2026-01-15 05:44:00,SU00000201,Mobile,5
10,E00000011,U000002,onboarding_start,2026-01-16 05:55:00,SU00000201,Mobile,5
11,E00000012,U000002,onboarding_complete,2026-01-17 12:11:00,SU00000201,Mobile,12
12,E00000013,U000003,signup,2026-03-13 02:08:00,SU00000301,Mobile,2
13,E00000014,U000003,onboarding_start,2026-03-13 23:29:00,SU00000301,Mobile,23


In [141]:
funnel_counts = {
    'Signup': events_clean[
        events_clean['event_name'] == 'signup'
    ]['user_id'].nunique(),

    'Onboarding Start': events_clean[
        events_clean['event_name'] == 'onboarding_start'
    ]['user_id'].nunique(),

    'Onboarding Complete': events_clean[
        events_clean['event_name'] == 'onboarding_complete'
    ]['user_id'].nunique(),

    'Core Action': events_clean[
        events_clean['event_name'] == 'core_action'
    ]['user_id'].nunique()
}

funnel_counts

{'Signup': 100000,
 'Onboarding Start': 100000,
 'Onboarding Complete': 74876,
 'Core Action': 36792}

In [142]:
signup_users = funnel_counts['Signup']

for stage, count in funnel_counts.items():
    conversion = count / signup_users * 100
    print(f"{stage}: {count:,} users ({conversion:.2f}%)")

Signup: 100,000 users (100.00%)
Onboarding Start: 100,000 users (100.00%)
Onboarding Complete: 74,876 users (74.88%)
Core Action: 36,792 users (36.79%)


In [143]:
started_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'onboarding_start',
        'user_id'
    ]
)

completed_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'onboarding_complete',
        'user_id'
    ]
)

completed_without_start = completed_users - started_users

len(completed_without_start)

0

In [144]:
core_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'core_action',
        'user_id'
    ]
)

core_without_completion = core_users - completed_users

len(core_without_completion)

0

In [145]:
print("Users:", users_clean.shape)
print("Unique users:", users_clean['user_id'].nunique())

users_clean.head()

Users: (100000, 6)
Unique users: 100000


,user_id,signup_date,country,device_type,age_group,acquisition_channel
0,U000001,2026-02-21,USA,Desktop,45-54,Paid Search
1,U000002,2026-01-15,Canada,Mobile,55+,Paid Search
2,U000003,2026-03-13,India,Mobile,25-34,Organic
3,U000004,2026-03-02,USA,Mobile,45-54,Social
4,U000005,2026-01-21,UK,Desktop,18-24,Organic


In [146]:
country_dist = users_clean['country'].value_counts()

country_dist

country
India        34874
USA          24972
UK           11897
Canada        9995
Germany       9951
Australia     8111
Unknown        200
Name: count, dtype: int64

In [147]:
country_pct = (
    users_clean['country']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

country_pct

country
India        34.87
USA          24.97
UK           11.90
Canada        9.99
Germany       9.95
Australia     8.11
Unknown       0.20
Name: proportion, dtype: float64

In [148]:
device_dist = users_clean['device_type'].value_counts()

device_dist

device_type
Mobile     61794
Desktop    29994
Tablet      8112
Name: count, dtype: int64

In [149]:
device_pct = (
    users_clean['device_type']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

device_pct

device_type
Mobile     61.86
Desktop    30.02
Tablet      8.12
Name: proportion, dtype: float64

In [150]:
age_dist = users_clean['age_group'].value_counts()

age_dist

age_group
25-34    36016
35-44    23121
18-24    19931
45-54    13982
55+       6950
Name: count, dtype: int64

In [151]:
age_pct = (
    users_clean['age_group']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

age_pct

age_group
25-34    36.02
35-44    23.12
18-24    19.93
45-54    13.98
55+       6.95
Name: proportion, dtype: float64

In [152]:
channel_dist = users_clean['acquisition_channel'].value_counts()

channel_dist

acquisition_channel
Organic        31885
Paid Search    24092
Social         18061
Referral       15997
Email           9965
Name: count, dtype: int64

In [153]:
channel_pct = (
    users_clean['acquisition_channel']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

channel_pct

acquisition_channel
Organic        31.88
Paid Search    24.09
Social         18.06
Referral       16.00
Email           9.96
Name: proportion, dtype: float64

In [154]:
experiment_dist = experiments_clean['experiment_group'].value_counts()

experiment_dist

experiment_group
Treatment    50014
Control      49986
Name: count, dtype: int64

In [155]:
experiment_pct = (
    experiments_clean['experiment_group']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

experiment_pct

experiment_group
Treatment    50.01
Control      49.99
Name: proportion, dtype: float64

In [156]:
user_experiment = users_clean.merge(
    experiments_clean[
        ['user_id', 'experiment_group', 'variant_version', 'assignment_date']
    ],
    on='user_id',
    how='inner'
)

user_experiment.shape

(100000, 9)

In [157]:
experiments_check = experiments_clean.merge(
    users_clean[['user_id', 'signup_date']],
    on='user_id',
    how='left'
)

invalid_assignments = experiments_check[
    experiments_check['assignment_date'] < experiments_check['signup_date']
]

len(invalid_assignments)

50

In [158]:
experiments_clean = experiments_check[
    experiments_check['assignment_date'] >= experiments_check['signup_date']
].copy()

In [159]:
experiments_clean = experiments_clean.drop(columns=['signup_date'])

In [160]:
experiments_clean.shape

(99950, 5)

In [161]:
experiments_clean['user_id'].nunique()

99950

In [162]:
user_experiment = users_clean.merge(
    experiments_clean[
        ['user_id', 'experiment_group', 'variant_version', 'assignment_date']
    ],
    on='user_id',
    how='inner'
)

In [163]:
user_experiment.shape


(99950, 9)

In [164]:
group_counts = user_experiment['experiment_group'].value_counts()

group_counts

experiment_group
Treatment    49987
Control      49963
Name: count, dtype: int64

In [165]:
group_pct = (
    user_experiment['experiment_group']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

group_pct

experiment_group
Treatment    50.01
Control      49.99
Name: proportion, dtype: float64

In [166]:
device_balance = pd.crosstab(
    user_experiment['device_type'],
    user_experiment['experiment_group'],
    normalize='columns'
).mul(100).round(2)

device_balance

experiment_group,Control,Treatment
device_type,,
Desktop,29.96,30.10
Mobile,61.98,61.74
Tablet,8.07,8.17


In [167]:
country_balance = pd.crosstab(
    user_experiment['country'],
    user_experiment['experiment_group'],
    normalize='columns'
).mul(100).round(2)

country_balance

experiment_group,Control,Treatment
country,,
Australia,8.05,8.18
Canada,9.98,10.01
Germany,9.86,10.04
India,34.80,34.95
UK,12.01,11.79
USA,25.09,24.84
Unknown,0.21,0.19


In [168]:
channel_balance = pd.crosstab(
    user_experiment['acquisition_channel'],
    user_experiment['experiment_group'],
    normalize='columns'
).mul(100).round(2)

channel_balance

experiment_group,Control,Treatment
acquisition_channel,,
Email,9.89,10.03
Organic,32.15,31.62
Paid Search,23.85,24.32
Referral,15.99,16.01
Social,18.11,18.01


In [169]:
age_balance = pd.crosstab(
    user_experiment['age_group'],
    user_experiment['experiment_group'],
    normalize='columns'
).mul(100).round(2)

age_balance

experiment_group,Control,Treatment
age_group,,
18-24,20.10,19.77
25-34,35.94,36.09
35-44,23.28,22.96
45-54,13.88,14.09
55+,6.81,7.09


In [170]:
from scipy.stats import chi2_contingency

In [171]:
country_table = pd.crosstab(
    user_experiment['country'],
    user_experiment['experiment_group']
)

chi2, p_value, dof, expected = chi2_contingency(country_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)

Chi-square statistic: 3.4920210478906415
p-value: 0.7450309561808621


In [172]:
def chi_square_test(df, column):
    table = pd.crosstab(
        df[column],
        df['experiment_group']
    )
    
    chi2, p_value, dof, expected = chi2_contingency(table)
    
    print(f"\n{column}")
    print(f"Chi-square: {chi2:.4f}")
    print(f"p-value: {p_value:.6f}")

In [173]:
chi_square_test(user_experiment, 'country')
chi_square_test(user_experiment, 'device_type')
chi_square_test(user_experiment, 'age_group')
chi_square_test(user_experiment, 'acquisition_channel')


country
Chi-square: 3.4920
p-value: 0.745031

device_type
Chi-square: 0.7057
p-value: 0.702669

age_group
Chi-square: 6.0560
p-value: 0.195006

acquisition_channel
Chi-square: 5.1712
p-value: 0.270182


In [174]:
funnel_counts = {
    'Signup': events_clean.loc[
        events_clean['event_name'] == 'signup',
        'user_id'
    ].nunique(),

    'Onboarding Start': events_clean.loc[
        events_clean['event_name'] == 'onboarding_start',
        'user_id'
    ].nunique(),

    'Onboarding Complete': events_clean.loc[
        events_clean['event_name'] == 'onboarding_complete',
        'user_id'
    ].nunique(),

    'Core Action': events_clean.loc[
        events_clean['event_name'] == 'core_action',
        'user_id'
    ].nunique()
}

funnel_counts

{'Signup': 100000,
 'Onboarding Start': 100000,
 'Onboarding Complete': 74876,
 'Core Action': 36792}

In [175]:
for stage, count in funnel_counts.items():
    print(f"{stage}: {count:,}")

Signup: 100,000
Onboarding Start: 100,000
Onboarding Complete: 74,876
Core Action: 36,792


In [176]:
signup_users = funnel_counts['Signup']

for stage, count in funnel_counts.items():
    conversion = count / signup_users * 100
    print(f"{stage}: {count:,} users ({conversion:.2f}%)")

Signup: 100,000 users (100.00%)
Onboarding Start: 100,000 users (100.00%)
Onboarding Complete: 74,876 users (74.88%)
Core Action: 36,792 users (36.79%)


In [177]:
signup_users_set = set(
    events_clean.loc[
        events_clean['event_name'] == 'signup',
        'user_id'
    ]
)

onboarding_start_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'onboarding_start',
        'user_id'
    ]
)

onboarding_complete_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'onboarding_complete',
        'user_id'
    ]
)

core_action_users = set(
    events_clean.loc[
        events_clean['event_name'] == 'core_action',
        'user_id'
    ]
)

In [178]:
funnel = {
    'Signup': len(signup_users_set),

    'Onboarding Start': len(
        signup_users_set & onboarding_start_users
    ),

    'Onboarding Complete': len(
        signup_users_set
        & onboarding_start_users
        & onboarding_complete_users
    ),

    'Core Action': len(
        signup_users_set
        & onboarding_start_users
        & onboarding_complete_users
        & core_action_users
    )
}

funnel

{'Signup': 100000,
 'Onboarding Start': 100000,
 'Onboarding Complete': 74876,
 'Core Action': 36792}

In [179]:
stages = list(funnel.keys())

for i in range(len(stages)):
    stage = stages[i]
    users = funnel[stage]

    if i == 0:
        rate = 100
    else:
        previous_users = funnel[stages[i-1]]
        rate = users / previous_users * 100

    print(
        f"{stage}: {users:,} users | "
        f"{rate:.2f}% conversion from previous stage"
    )

Signup: 100,000 users | 100.00% conversion from previous stage
Onboarding Start: 100,000 users | 100.00% conversion from previous stage
Onboarding Complete: 74,876 users | 74.88% conversion from previous stage
Core Action: 36,792 users | 49.14% conversion from previous stage


In [180]:
onboarding_complete = (
    events_clean[
        events_clean['event_name'] == 'onboarding_complete'
    ]
    .groupby('user_id')['event_timestamp']
    .min()
    .reset_index()
    .rename(columns={
        'event_timestamp': 'onboarding_complete_time'
    })
)

In [181]:
core_actions = events_clean[
    events_clean['event_name'] == 'core_action'
][['user_id', 'event_timestamp']].copy()

core_actions = core_actions.rename(
    columns={'event_timestamp': 'core_action_time'}
)

In [182]:
activation_df = users_clean[
    ['user_id', 'signup_date']
].copy()

In [183]:
activation_df = activation_df.merge(
    onboarding_complete,
    on='user_id',
    how='left'
)

In [184]:
activation_events = activation_df.merge(
    core_actions,
    on='user_id',
    how='left'
)

In [185]:
activation_events['days_from_signup'] = (
    activation_events['core_action_time']
    - activation_events['signup_date']
).dt.total_seconds() / (24 * 60 * 60)

In [186]:
core_within_7_days = activation_events[
    (activation_events['days_from_signup'] >= 0) &
    (activation_events['days_from_signup'] <= 7)
]

In [187]:
core_7d_users = set(
    core_within_7_days['user_id']
)

In [188]:
completed_users = set(
    onboarding_complete['user_id']
)

In [189]:
activated_users = completed_users & core_7d_users

In [190]:
len(activated_users)

35909

In [191]:
eligible_users = set(
    experiments_clean['user_id']
)

In [192]:
activated_eligible_users = activated_users & eligible_users

activation_rate = (
    len(activated_eligible_users)
    / len(eligible_users)
    * 100
)

print("Eligible users:", len(eligible_users))
print("Activated users:", len(activated_eligible_users))
print(f"7-Day Activation Rate: {activation_rate:.2f}%")

Eligible users: 99950
Activated users: 35891
7-Day Activation Rate: 35.91%


In [193]:
activation_df['onboarding_days_from_signup'] = (
    activation_df['onboarding_complete_time']
    - activation_df['signup_date']
).dt.total_seconds() / (24 * 60 * 60)

In [194]:
onboarding_7d_users = set(
    activation_df.loc[
        (activation_df['onboarding_days_from_signup'] >= 0) &
        (activation_df['onboarding_days_from_signup'] <= 7),
        'user_id'
    ]
)

In [195]:
activated_users = onboarding_7d_users & core_7d_users

In [196]:
activated_eligible_users = activated_users & eligible_users

In [197]:
activation_rate = (
    len(activated_eligible_users)
    / len(eligible_users)
    * 100
)

print("Eligible users:", len(eligible_users))
print("Activated users:", len(activated_eligible_users))
print(f"7-Day Activation Rate: {activation_rate:.2f}%")

Eligible users: 99950
Activated users: 35891
7-Day Activation Rate: 35.91%


In [198]:
# A/B Test Analysis

In [201]:
experiment_activation = user_experiment.merge(
    activation_status,
    on='user_id',
    how='inner'
)

print("Rows:", len(experiment_activation))
print("Columns:", experiment_activation.shape[1])

Rows: 99950
Columns: 10


In [202]:
experiment_activation.head()

,user_id,signup_date,country,device_type,age_group,acquisition_channel,experiment_group,variant_version,assignment_date,activated
0,U000001,2026-02-21,USA,Desktop,45-54,Paid Search,Treatment,v2,2026-02-21,1
1,U000002,2026-01-15,Canada,Mobile,55+,Paid Search,Treatment,v2,2026-01-15,0
2,U000003,2026-03-13,India,Mobile,25-34,Organic,Treatment,v2,2026-03-13,0
3,U000004,2026-03-02,USA,Mobile,45-54,Social,Control,v1,2026-03-02,1
4,U000005,2026-01-21,UK,Desktop,18-24,Organic,Control,v1,2026-01-21,0


In [203]:
print(
    experiment_activation['experiment_group']
    .value_counts()
)

print()

print(
    experiment_activation['activated']
    .value_counts()
)

experiment_group
Treatment    49987
Control      49963
Name: count, dtype: int64

activated
0    64059
1    35891
Name: count, dtype: int64


In [204]:
activation_by_group = (
    experiment_activation
    .groupby('experiment_group')['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
)

activation_by_group['activation_rate'] *= 100

activation_by_group

,users,activated_users,activation_rate
experiment_group,,,
Control,49963,16710,33.444749
Treatment,49987,19181,38.371977


In [205]:
control_rate = activation_by_group.loc[
    'Control', 'activation_rate'
]

treatment_rate = activation_by_group.loc[
    'Treatment', 'activation_rate'
]

print(f"Control Activation Rate: {control_rate:.2f}%")
print(f"Treatment Activation Rate: {treatment_rate:.2f}%")

Control Activation Rate: 33.44%
Treatment Activation Rate: 38.37%


In [206]:
absolute_lift = treatment_rate - control_rate

print(f"Absolute Lift: {absolute_lift:.2f} percentage points")

Absolute Lift: 4.93 percentage points


In [207]:
relative_lift = (
    (treatment_rate - control_rate)
    / control_rate
) * 100

print(f"Relative Lift: {relative_lift:.2f}%")

Relative Lift: 14.73%


In [208]:
ab_summary = pd.DataFrame({
    'Metric': [
        'Users',
        'Activated Users',
        'Activation Rate (%)'
    ],
    'Control': [
        activation_by_group.loc['Control', 'users'],
        activation_by_group.loc['Control', 'activated_users'],
        control_rate
    ],
    'Treatment': [
        activation_by_group.loc['Treatment', 'users'],
        activation_by_group.loc['Treatment', 'activated_users'],
        treatment_rate
    ]
})

ab_summary

,Metric,Control,Treatment
0,Users,49963.000000,49987.000000
1,Activated Users,16710.000000,19181.000000
2,Activation Rate (%),33.444749,38.371977


In [209]:
print(f"Absolute Lift: {absolute_lift:.2f} percentage points")
print(f"Relative Lift: {relative_lift:.2f}%")

Absolute Lift: 4.93 percentage points
Relative Lift: 14.73%


In [210]:
from statsmodels.stats.proportion import proportions_ztest

control_activated = int(
    activation_by_group.loc['Control', 'activated_users']
)

control_users = int(
    activation_by_group.loc['Control', 'users']
)

treatment_activated = int(
    activation_by_group.loc['Treatment', 'activated_users']
)

treatment_users = int(
    activation_by_group.loc['Treatment', 'users']
)

successes = [
    control_activated,
    treatment_activated
]

sample_sizes = [
    control_users,
    treatment_users
]

z_stat, p_value = proportions_ztest(
    successes,
    sample_sizes
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.10f}")

Z-statistic: -16.2354
P-value: 0.0000000000


In [211]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    control_activated,
    control_users,
    treatment_activated,
    treatment_users,
    method='wald'
)

print(f"95% CI Lower: {ci_low * 100:.2f}%")
print(f"95% CI Upper: {ci_high * 100:.2f}%")

95% CI Lower: -5.52%
95% CI Upper: -4.33%


In [212]:
events_experiment = events_clean.merge(
    experiments_clean[
        ['user_id', 'experiment_group']
    ],
    on='user_id',
    how='inner'
)

events_experiment.shape

(432315, 8)

In [213]:
def get_funnel_by_group(df):
    
    stages = [
        'signup',
        'onboarding_start',
        'onboarding_complete',
        'core_action'
    ]
    
    results = []
    
    for group in ['Control', 'Treatment']:
        
        group_df = df[
            df['experiment_group'] == group
        ]
        
        signup_users = set(
            group_df.loc[
                group_df['event_name'] == 'signup',
                'user_id'
            ]
        )
        
        start_users = set(
            group_df.loc[
                group_df['event_name'] == 'onboarding_start',
                'user_id'
            ]
        )
        
        complete_users = set(
            group_df.loc[
                group_df['event_name'] == 'onboarding_complete',
                'user_id'
            ]
        )
        
        core_users = set(
            group_df.loc[
                group_df['event_name'] == 'core_action',
                'user_id'
            ]
        )
        
        funnel = {
            'Signup': len(signup_users),
            
            'Onboarding Start': len(
                signup_users & start_users
            ),
            
            'Onboarding Complete': len(
                signup_users
                & start_users
                & complete_users
            ),
            
            'Core Action': len(
                signup_users
                & start_users
                & complete_users
                & core_users
            )
        }
        
        for stage, users in funnel.items():
            results.append({
                'Experiment Group': group,
                'Stage': stage,
                'Users': users
            })
    
    return pd.DataFrame(results)

In [214]:
funnel_by_group = get_funnel_by_group(
    events_experiment
)

funnel_by_group

,Experiment Group,Stage,Users
0,Control,Signup,49963
1,Control,Onboarding Start,49963
2,Control,Onboarding Complete,35935
3,Control,Core Action,17142
4,Treatment,Signup,49987
5,Treatment,Onboarding Start,49987
6,Treatment,Onboarding Complete,38899
7,Treatment,Core Action,19631


In [215]:
funnel_by_group['Conversion (%)'] = 0.0

for group in ['Control', 'Treatment']:
    
    mask = (
        funnel_by_group['Experiment Group']
        == group
    )
    
    signup_count = funnel_by_group.loc[
        mask & (funnel_by_group['Stage'] == 'Signup'),
        'Users'
    ].iloc[0]
    
    funnel_by_group.loc[mask, 'Conversion (%)'] = (
        funnel_by_group.loc[mask, 'Users']
        / signup_count
        * 100
    )

funnel_by_group

,Experiment Group,Stage,Users,Conversion (%)
0,Control,Signup,49963,100.000000
1,Control,Onboarding Start,49963,100.000000
2,Control,Onboarding Complete,35935,71.923223
3,Control,Core Action,17142,34.309389
4,Treatment,Signup,49987,100.000000
5,Treatment,Onboarding Start,49987,100.000000
6,Treatment,Onboarding Complete,38899,77.818233
7,Treatment,Core Action,19631,39.272211


In [216]:
funnel_by_group['Stage Conversion (%)'] = 0.0

stage_order = [
    'Signup',
    'Onboarding Start',
    'Onboarding Complete',
    'Core Action'
]

for group in ['Control', 'Treatment']:
    
    mask = (
        funnel_by_group['Experiment Group']
        == group
    )
    
    group_data = funnel_by_group[mask].copy()
    
    group_data = group_data.set_index('Stage')
    
    for i, stage in enumerate(stage_order):
        
        current_users = group_data.loc[
            stage, 'Users'
        ]
        
        if i == 0:
            conversion = 100
        else:
            previous_stage = stage_order[i - 1]
            
            previous_users = group_data.loc[
                previous_stage, 'Users'
            ]
            
            conversion = (
                current_users
                / previous_users
                * 100
            )
        
        funnel_by_group.loc[
            mask & (funnel_by_group['Stage'] == stage),
            'Stage Conversion (%)'
        ] = conversion

funnel_by_group

,Experiment Group,Stage,Users,Conversion (%),Stage Conversion (%)
0,Control,Signup,49963,100.000000,100.000000
1,Control,Onboarding Start,49963,100.000000,100.000000
2,Control,Onboarding Complete,35935,71.923223,71.923223
3,Control,Core Action,17142,34.309389,47.702797
4,Treatment,Signup,49987,100.000000,100.000000
5,Treatment,Onboarding Start,49987,100.000000,100.000000
6,Treatment,Onboarding Complete,38899,77.818233,77.818233
7,Treatment,Core Action,19631,39.272211,50.466593


In [217]:
funnel_pivot = funnel_by_group.pivot(
    index='Stage',
    columns='Experiment Group',
    values='Conversion (%)'
)

funnel_pivot['Absolute Lift (pp)'] = (
    funnel_pivot['Treatment']
    - funnel_pivot['Control']
)

funnel_pivot['Relative Lift (%)'] = (
    funnel_pivot['Absolute Lift (pp)']
    / funnel_pivot['Control']
) * 100

funnel_pivot

Experiment Group,Control,Treatment,Absolute Lift (pp),Relative Lift (%)
Stage,,,,
Core Action,34.309389,39.272211,4.962822,14.464909
Onboarding Complete,71.923223,77.818233,5.895010,8.196253
Onboarding Start,100.000000,100.000000,0.000000,0.000000
Signup,100.000000,100.000000,0.000000,0.000000


In [218]:
experiment_activation[
    [
        'user_id',
        'experiment_group',
        'device_type',
        'country',
        'age_group',
        'acquisition_channel',
        'activated'
    ]
].head()

,user_id,experiment_group,device_type,country,age_group,acquisition_channel,activated
0,U000001,Treatment,Desktop,USA,45-54,Paid Search,1
1,U000002,Treatment,Mobile,Canada,55+,Paid Search,0
2,U000003,Treatment,Mobile,India,25-34,Organic,0
3,U000004,Control,Mobile,USA,45-54,Social,1
4,U000005,Control,Desktop,UK,18-24,Organic,0


In [219]:
device_activation = (
    experiment_activation
    .groupby(['device_type', 'experiment_group'])['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
    .reset_index()
)

device_activation['activation_rate'] *= 100

device_activation

,device_type,experiment_group,users,activated_users,activation_rate
0,Desktop,Control,14951,4976,33.282055
1,Desktop,Treatment,15030,5689,37.850965
2,Mobile,Control,30934,10363,33.500356
3,Mobile,Treatment,30831,11891,38.568324
4,Tablet,Control,4026,1353,33.606557
5,Tablet,Treatment,4078,1581,38.769004


In [220]:
device_pivot = device_activation.pivot(
    index='device_type',
    columns='experiment_group',
    values='activation_rate'
)

device_pivot['Absolute Lift (pp)'] = (
    device_pivot['Treatment']
    - device_pivot['Control']
)

device_pivot['Relative Lift (%)'] = (
    device_pivot['Absolute Lift (pp)']
    / device_pivot['Control']
) * 100

device_pivot.round(2)

experiment_group,Control,Treatment,Absolute Lift (pp),Relative Lift (%)
device_type,,,,
Desktop,33.28,37.85,4.57,13.73
Mobile,33.50,38.57,5.07,15.13
Tablet,33.61,38.77,5.16,15.36


In [221]:
country_activation = (
    experiment_activation
    .groupby(['country', 'experiment_group'])['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
    .reset_index()
)

country_activation['activation_rate'] *= 100

country_activation

,country,experiment_group,users,activated_users,activation_rate
0,Australia,Control,4020,1383,34.402985
1,Australia,Treatment,4088,1545,37.793542
2,Canada,Control,4987,1643,32.945659
3,Canada,Treatment,5004,1914,38.249400
4,Germany,Control,4928,1646,33.400974
5,Germany,Treatment,5019,1928,38.414027
6,India,Control,17387,5829,33.525047
7,India,Treatment,17470,6717,38.448769
8,UK,Control,5999,1922,32.038673
9,UK,Treatment,5891,2280,38.703106


In [222]:
country_pivot = country_activation.pivot(
    index='country',
    columns='experiment_group',
    values='activation_rate'
)

country_pivot['Absolute Lift (pp)'] = (
    country_pivot['Treatment']
    - country_pivot['Control']
)

country_pivot['Relative Lift (%)'] = (
    country_pivot['Absolute Lift (pp)']
    / country_pivot['Control']
) * 100

country_pivot.round(2)

experiment_group,Control,Treatment,Absolute Lift (pp),Relative Lift (%)
country,,,,
Australia,34.40,37.79,3.39,9.86
Canada,32.95,38.25,5.30,16.10
Germany,33.40,38.41,5.01,15.01
India,33.53,38.45,4.92,14.69
UK,32.04,38.70,6.66,20.80
USA,33.89,38.34,4.45,13.12
Unknown,36.54,37.50,0.96,2.63


In [223]:
channel_activation = (
    experiment_activation
    .groupby(
        ['acquisition_channel', 'experiment_group']
    )['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
    .reset_index()
)

channel_activation['activation_rate'] *= 100

channel_activation

,acquisition_channel,experiment_group,users,activated_users,activation_rate
0,Email,Control,4943,1662,33.623306
1,Email,Treatment,5014,1889,37.674511
2,Organic,Control,16065,5311,33.059446
3,Organic,Treatment,15807,5992,37.907256
4,Paid Search,Control,11916,4012,33.669016
5,Paid Search,Treatment,12159,4711,38.744963
6,Referral,Control,7989,2701,33.808987
7,Referral,Treatment,8003,3141,39.247782
8,Social,Control,9050,3024,33.414365
9,Social,Treatment,9004,3448,38.294092


In [224]:
channel_pivot = channel_activation.pivot(
    index='acquisition_channel',
    columns='experiment_group',
    values='activation_rate'
)

channel_pivot['Absolute Lift (pp)'] = (
    channel_pivot['Treatment']
    - channel_pivot['Control']
)

channel_pivot['Relative Lift (%)'] = (
    channel_pivot['Absolute Lift (pp)']
    / channel_pivot['Control']
) * 100

channel_pivot.round(2)

experiment_group,Control,Treatment,Absolute Lift (pp),Relative Lift (%)
acquisition_channel,,,,
Email,33.62,37.67,4.05,12.05
Organic,33.06,37.91,4.85,14.66
Paid Search,33.67,38.74,5.08,15.08
Referral,33.81,39.25,5.44,16.09
Social,33.41,38.29,4.88,14.60


In [225]:
age_activation = (
    experiment_activation
    .groupby(
        ['age_group', 'experiment_group']
    )['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
    .reset_index()
)

age_activation['activation_rate'] *= 100

age_pivot = age_activation.pivot(
    index='age_group',
    columns='experiment_group',
    values='activation_rate'
)

age_pivot['Absolute Lift (pp)'] = (
    age_pivot['Treatment']
    - age_pivot['Control']
)

age_pivot['Relative Lift (%)'] = (
    age_pivot['Absolute Lift (pp)']
    / age_pivot['Control']
) * 100

age_pivot.round(2)

experiment_group,Control,Treatment,Absolute Lift (pp),Relative Lift (%)
age_group,,,,
18-24,33.48,37.97,4.49,13.41
25-34,33.61,37.98,4.37,13.01
35-44,33.45,38.71,5.26,15.72
45-54,32.68,39.41,6.73,20.58
55+,34.00,38.31,4.31,12.68


In [226]:
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd
import numpy as np

def segment_ab_test(df, segment_column):
    
    results = []

    segments = df[segment_column].dropna().unique()

    for segment in sorted(segments):

        segment_df = df[df[segment_column] == segment]

        control = segment_df[
            segment_df['experiment_group'] == 'Control'
        ]

        treatment = segment_df[
            segment_df['experiment_group'] == 'Treatment'
        ]

        control_users = len(control)
        treatment_users = len(treatment)

        control_activated = control['activated'].sum()
        treatment_activated = treatment['activated'].sum()

        control_rate = control_activated / control_users
        treatment_rate = treatment_activated / treatment_users

        # Two-proportion z-test
        successes = np.array([
            control_activated,
            treatment_activated
        ])

        observations = np.array([
            control_users,
            treatment_users
        ])

        z_stat, p_value = proportions_ztest(
            successes,
            observations
        )

        absolute_lift = (
            treatment_rate - control_rate
        ) * 100

        relative_lift = (
            absolute_lift /
            (control_rate * 100)
        ) * 100

        results.append({
            'Segment': segment,
            'Control Users': control_users,
            'Treatment Users': treatment_users,
            'Control Rate (%)': control_rate * 100,
            'Treatment Rate (%)': treatment_rate * 100,
            'Absolute Lift (pp)': absolute_lift,
            'Relative Lift (%)': relative_lift,
            'Z-Statistic': z_stat,
            'P-Value': p_value
        })

    return pd.DataFrame(results)

In [228]:
# Create a complete analysis dataset
segment_analysis = experiment_activation.merge(
    users_clean[
        ['user_id',
         'country',
         'device_type',
         'age_group',
         'acquisition_channel']
    ],
    on='user_id',
    how='left'
)

segment_analysis.shape

(99950, 14)

In [229]:
segment_analysis.columns

Index(['user_id', 'signup_date', 'country_x', 'device_type_x', 'age_group_x',
       'acquisition_channel_x', 'experiment_group', 'variant_version',
       'assignment_date', 'activated', 'country_y', 'device_type_y',
       'age_group_y', 'acquisition_channel_y'],
      dtype='object')

In [231]:
print("experiment_activation columns:")
print(experiment_activation.columns.tolist())

print("\nusers_clean columns:")
print(users_clean.columns.tolist())

print("\nsegment_analysis columns:")
print(segment_analysis.columns.tolist())

experiment_activation columns:
['user_id', 'signup_date', 'country', 'device_type', 'age_group', 'acquisition_channel', 'experiment_group', 'variant_version', 'assignment_date', 'activated']

users_clean columns:
['user_id', 'signup_date', 'country', 'device_type', 'age_group', 'acquisition_channel']

segment_analysis columns:
['user_id', 'signup_date', 'country_x', 'device_type_x', 'age_group_x', 'acquisition_channel_x', 'experiment_group', 'variant_version', 'assignment_date', 'activated', 'country_y', 'device_type_y', 'age_group_y', 'acquisition_channel_y']


In [232]:
print(segment_analysis.head())

   user_id signup_date country_x device_type_x age_group_x  \
0  U000001  2026-02-21       USA       Desktop       45-54   
1  U000002  2026-01-15    Canada        Mobile         55+   
2  U000003  2026-03-13     India        Mobile       25-34   
3  U000004  2026-03-02       USA        Mobile       45-54   
4  U000005  2026-01-21        UK       Desktop       18-24   

  acquisition_channel_x experiment_group variant_version assignment_date  \
0           Paid Search        Treatment              v2      2026-02-21   
1           Paid Search        Treatment              v2      2026-01-15   
2               Organic        Treatment              v2      2026-03-13   
3                Social          Control              v1      2026-03-02   
4               Organic          Control              v1      2026-01-21   

   activated country_y device_type_y age_group_y acquisition_channel_y  
0          1       USA       Desktop       45-54           Paid Search  
1          0    Canada  

In [233]:
del segment_analysis

In [234]:
experiment_activation[
    [
        'user_id',
        'experiment_group',
        'activated',
        'country',
        'device_type',
        'age_group',
        'acquisition_channel'
    ]
].head()

,user_id,experiment_group,activated,country,device_type,age_group,acquisition_channel
0,U000001,Treatment,1,USA,Desktop,45-54,Paid Search
1,U000002,Treatment,0,Canada,Mobile,55+,Paid Search
2,U000003,Treatment,0,India,Mobile,25-34,Organic
3,U000004,Control,1,USA,Mobile,45-54,Social
4,U000005,Control,0,UK,Desktop,18-24,Organic


In [237]:
experiment_activation[
    [
        'country',
        'device_type',
        'age_group',
        'acquisition_channel'
    ]
].isna().sum()

country                0
device_type            0
age_group              0
acquisition_channel    0
dtype: int64

In [236]:
experiment_activation['device_type'] = (
    experiment_activation['device_type']
    .fillna('Unknown')
)

In [238]:
experiment_activation['device_type'].value_counts()

device_type
Mobile     61765
Desktop    29981
Tablet      8104
Unknown      100
Name: count, dtype: int64

In [239]:
device_stats = segment_ab_test(
    experiment_activation,
    'device_type'
)

device_stats.round(4)

,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),Z-Statistic,P-Value
0,Desktop,14951,15030,33.2821,37.8510,4.5689,13.7278,-8.2625,0.000
1,Mobile,30934,30831,33.5004,38.5683,5.0680,15.1281,-13.1176,0.000
2,Tablet,4026,4078,33.6066,38.7690,5.1624,15.3614,-4.8349,0.000
3,Unknown,52,48,34.6154,41.6667,7.0513,20.3704,-0.7258,0.468


In [240]:
country_stats = segment_ab_test(
    experiment_activation,
    'country'
)

country_stats.round(4)

,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),Z-Statistic,P-Value
0,Australia,4020,4088,34.4030,37.7935,3.3906,9.8554,-3.1779,0.0015
1,Canada,4987,5004,32.9457,38.2494,5.3037,16.0985,-5.5358,0.0000
2,Germany,4928,5019,33.4010,38.4140,5.0131,15.0087,-5.2101,0.0000
3,India,17387,17470,33.5250,38.4488,4.9237,14.6867,-9.5760,0.0000
4,UK,5999,5891,32.0387,38.7031,6.6644,20.8012,-7.6007,0.0000
5,USA,12538,12419,33.8890,38.3364,4.4474,13.1236,-7.3141,0.0000
6,Unknown,104,96,36.5385,37.5000,0.9615,2.6316,-0.1407,0.8881


In [241]:
channel_stats = segment_ab_test(
    experiment_activation,
    'acquisition_channel'
)

channel_stats.round(4)

,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),Z-Statistic,P-Value
0,Email,4943,5014,33.6233,37.6745,4.0512,12.0488,-4.2196,0.0
1,Organic,16065,15807,33.0594,37.9073,4.8478,14.6639,-9.0451,0.0
2,Paid Search,11916,12159,33.6690,38.7450,5.0759,15.0760,-8.1922,0.0
3,Referral,7989,8003,33.8090,39.2478,5.4388,16.0868,-7.1419,0.0
4,Social,9050,9004,33.4144,38.2941,4.8797,14.6037,-6.8362,0.0


In [242]:
age_stats = segment_ab_test(
    experiment_activation,
    'age_group'
)

age_stats.round(4)

,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),Z-Statistic,P-Value
0,18-24,10041,9883,33.4827,37.9743,4.4916,13.4146,-6.6157,0.0000
1,25-34,17956,18042,33.6099,37.9836,4.3737,13.0130,-8.6544,0.0000
2,35-44,11629,11478,33.4509,38.7088,5.2580,15.7185,-8.3223,0.0000
3,45-54,6934,7042,32.6796,39.4064,6.7269,20.5843,-8.2802,0.0000
4,55+,3403,3542,33.9994,38.3117,4.3123,12.6834,-3.7382,0.0002


In [243]:
segment_results = pd.concat([
    device_stats.assign(Segment_Type='Device'),
    country_stats.assign(Segment_Type='Country'),
    channel_stats.assign(Segment_Type='Acquisition Channel'),
    age_stats.assign(Segment_Type='Age Group')
], ignore_index=True)

segment_results.shape

(21, 10)

In [244]:
from statsmodels.stats.multitest import multipletests

reject, adjusted_pvalues, _, _ = multipletests(
    segment_results['P-Value'],
    alpha=0.05,
    method='fdr_bh'
)

segment_results['Adjusted P-Value'] = adjusted_pvalues

segment_results['Significant After FDR'] = np.where(
    reject,
    'Yes',
    'No'
)

In [245]:
final_segment_results = segment_results[
    [
        'Segment_Type',
        'Segment',
        'Control Users',
        'Treatment Users',
        'Control Rate (%)',
        'Treatment Rate (%)',
        'Absolute Lift (pp)',
        'Relative Lift (%)',
        'P-Value',
        'Adjusted P-Value',
        'Significant After FDR'
    ]
].sort_values(
    'Adjusted P-Value'
)

final_segment_results.round(4)


,Segment_Type,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),P-Value,Adjusted P-Value,Significant After FDR
1,Device,Mobile,30934,30831,33.5004,38.5683,5.0680,15.1281,0.0000,0.0000,Yes
7,Country,India,17387,17470,33.5250,38.4488,4.9237,14.6867,0.0000,0.0000,Yes
12,Acquisition Channel,Organic,16065,15807,33.0594,37.9073,4.8478,14.6639,0.0000,0.0000,Yes
17,Age Group,25-34,17956,18042,33.6099,37.9836,4.3737,13.0130,0.0000,0.0000,Yes
18,Age Group,35-44,11629,11478,33.4509,38.7088,5.2580,15.7185,0.0000,0.0000,Yes
0,Device,Desktop,14951,15030,33.2821,37.8510,4.5689,13.7278,0.0000,0.0000,Yes
19,Age Group,45-54,6934,7042,32.6796,39.4064,6.7269,20.5843,0.0000,0.0000,Yes
13,Acquisition Channel,Paid Search,11916,12159,33.6690,38.7450,5.0759,15.0760,0.0000,0.0000,Yes
8,Country,UK,5999,5891,32.0387,38.7031,6.6644,20.8012,0.0000,0.0000,Yes
9,Country,USA,12538,12419,33.8890,38.3364,4.4474,13.1236,0.0000,0.0000,Yes


In [246]:
final_segment_results[
    final_segment_results['Significant After FDR'] == 'Yes'
].round(4)

,Segment_Type,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),P-Value,Adjusted P-Value,Significant After FDR
1,Device,Mobile,30934,30831,33.5004,38.5683,5.0680,15.1281,0.0000,0.0000,Yes
7,Country,India,17387,17470,33.5250,38.4488,4.9237,14.6867,0.0000,0.0000,Yes
12,Acquisition Channel,Organic,16065,15807,33.0594,37.9073,4.8478,14.6639,0.0000,0.0000,Yes
17,Age Group,25-34,17956,18042,33.6099,37.9836,4.3737,13.0130,0.0000,0.0000,Yes
18,Age Group,35-44,11629,11478,33.4509,38.7088,5.2580,15.7185,0.0000,0.0000,Yes
0,Device,Desktop,14951,15030,33.2821,37.8510,4.5689,13.7278,0.0000,0.0000,Yes
19,Age Group,45-54,6934,7042,32.6796,39.4064,6.7269,20.5843,0.0000,0.0000,Yes
13,Acquisition Channel,Paid Search,11916,12159,33.6690,38.7450,5.0759,15.0760,0.0000,0.0000,Yes
8,Country,UK,5999,5891,32.0387,38.7031,6.6644,20.8012,0.0000,0.0000,Yes
9,Country,USA,12538,12419,33.8890,38.3364,4.4474,13.1236,0.0000,0.0000,Yes


In [247]:
final_segment_results[
    final_segment_results['Significant After FDR'] == 'No'
].round(4)

,Segment_Type,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),P-Value,Adjusted P-Value,Significant After FDR
3,Device,Unknown,52,48,34.6154,41.6667,7.0513,20.3704,0.4680,0.4914,No
10,Country,Unknown,104,96,36.5385,37.5000,0.9615,2.6316,0.8881,0.8881,No


In [248]:
final_segment_results[
    final_segment_results['Significant After FDR'] == 'Yes'
].round(4)

,Segment_Type,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),P-Value,Adjusted P-Value,Significant After FDR
1,Device,Mobile,30934,30831,33.5004,38.5683,5.0680,15.1281,0.0000,0.0000,Yes
7,Country,India,17387,17470,33.5250,38.4488,4.9237,14.6867,0.0000,0.0000,Yes
12,Acquisition Channel,Organic,16065,15807,33.0594,37.9073,4.8478,14.6639,0.0000,0.0000,Yes
17,Age Group,25-34,17956,18042,33.6099,37.9836,4.3737,13.0130,0.0000,0.0000,Yes
18,Age Group,35-44,11629,11478,33.4509,38.7088,5.2580,15.7185,0.0000,0.0000,Yes
0,Device,Desktop,14951,15030,33.2821,37.8510,4.5689,13.7278,0.0000,0.0000,Yes
19,Age Group,45-54,6934,7042,32.6796,39.4064,6.7269,20.5843,0.0000,0.0000,Yes
13,Acquisition Channel,Paid Search,11916,12159,33.6690,38.7450,5.0759,15.0760,0.0000,0.0000,Yes
8,Country,UK,5999,5891,32.0387,38.7031,6.6644,20.8012,0.0000,0.0000,Yes
9,Country,USA,12538,12419,33.8890,38.3364,4.4474,13.1236,0.0000,0.0000,Yes


In [249]:
final_segment_results[
    final_segment_results['Significant After FDR'] == 'No'
].round(4)

,Segment_Type,Segment,Control Users,Treatment Users,Control Rate (%),Treatment Rate (%),Absolute Lift (pp),Relative Lift (%),P-Value,Adjusted P-Value,Significant After FDR
3,Device,Unknown,52,48,34.6154,41.6667,7.0513,20.3704,0.4680,0.4914,No
10,Country,Unknown,104,96,36.5385,37.5000,0.9615,2.6316,0.8881,0.8881,No


In [250]:
print(
    "Total segment tests:",
    len(final_segment_results)
)

print(
    "Significant after FDR:",
    (final_segment_results['Significant After FDR'] == 'Yes').sum()
)

print(
    "Not significant after FDR:",
    (final_segment_results['Significant After FDR'] == 'No').sum()
)

Total segment tests: 21
Significant after FDR: 19
Not significant after FDR: 2


In [251]:
final_segment_results.to_csv(
    'segment_ab_test_results.csv',
    index=False
)

In [252]:
delayed_assignment = experiment_activation[
    experiment_activation['assignment_date'] > 
    experiment_activation['signup_date']
].copy()

print("Delayed assignment users:", len(delayed_assignment))

Delayed assignment users: 2037


In [253]:
delayed_pct = (
    len(delayed_assignment) /
    len(experiment_activation)
) * 100

print(f"Delayed assignment percentage: {delayed_pct:.2f}%")

Delayed assignment percentage: 2.04%


In [254]:
pre_assignment_events = events_clean.merge(
    experiments_clean[['user_id', 'assignment_date']],
    on='user_id',
    how='inner'
)

pre_assignment_events['event_timestamp'] = pd.to_datetime(
    pre_assignment_events['event_timestamp']
)

pre_assignment_events['assignment_date'] = pd.to_datetime(
    pre_assignment_events['assignment_date']
)

pre_assignment_events = pre_assignment_events[
    pre_assignment_events['event_timestamp'] < 
    pre_assignment_events['assignment_date']
].copy()

print("Pre-assignment events:", len(pre_assignment_events))

Pre-assignment events: 3044


In [255]:
pre_assignment_users = (
    pre_assignment_events['user_id']
    .nunique()
)

print("Users with pre-assignment events:", pre_assignment_users)

Users with pre-assignment events: 2037


In [256]:
pre_assignment_user_pct = (
    pre_assignment_users /
    len(experiment_activation)
) * 100

print(
    f"Percentage of users with pre-assignment events: "
    f"{pre_assignment_user_pct:.2f}%"
)

Percentage of users with pre-assignment events: 2.04%


In [257]:
pre_assignment_events['event_name'].value_counts()

event_name
signup              2037
onboarding_start    1007
Name: count, dtype: int64

In [258]:
activation_events = [
    'onboarding_complete',
    'core_action'
]

pre_assignment_activation = pre_assignment_events[
    pre_assignment_events['event_name'].isin(
        activation_events
    )
]

print(
    "Pre-assignment activation events:",
    len(pre_assignment_activation)
)

Pre-assignment activation events: 0


In [259]:
print(
    "Users with pre-assignment activation:",
    pre_assignment_activation['user_id'].nunique()
)

Users with pre-assignment activation: 0


In [260]:
same_day_users = experiment_activation[
    experiment_activation['assignment_date'] ==
    experiment_activation['signup_date']
].copy()

print("Same-day assignment users:", len(same_day_users))

Same-day assignment users: 97913


In [261]:
same_day_summary = (
    same_day_users
    .groupby('experiment_group')['activated']
    .agg(
        users='count',
        activated_users='sum',
        activation_rate='mean'
    )
    .reset_index()
)

same_day_summary['activation_rate'] *= 100

same_day_summary.round(2)

,experiment_group,users,activated_users,activation_rate
0,Control,48939,16341,33.39
1,Treatment,48974,18776,38.34


In [262]:
print("Earliest event:",
      events_clean['event_timestamp'].min())

print("Latest event:",
      events_clean['event_timestamp'].max())

print("Latest signup:",
      experiment_activation['signup_date'].max())

Earliest event: 2026-01-01 00:00:00
Latest event: 2026-04-30 23:40:00
Latest signup: 2026-03-31 00:00:00


In [263]:
retention_users = experiment_activation[
    [
        'user_id',
        'signup_date',
        'experiment_group'
    ]
].copy()

retention_users['signup_date'] = pd.to_datetime(
    retention_users['signup_date']
)

observation_end = events_clean['event_timestamp'].max().normalize()

retention_users['d1_eligible'] = (
    retention_users['signup_date'] + pd.Timedelta(days=1)
    <= observation_end
)

retention_users['d7_eligible'] = (
    retention_users['signup_date'] + pd.Timedelta(days=7)
    <= observation_end
)

retention_users['d30_eligible'] = (
    retention_users['signup_date'] + pd.Timedelta(days=30)
    <= observation_end
)

retention_users[
    ['d1_eligible', 'd7_eligible', 'd30_eligible']
].sum()

d1_eligible     99950
d7_eligible     99950
d30_eligible    99950
dtype: int64

In [264]:
retention_events = [
    'core_action',
    'feature_view',
    'session_start',
    'subscription_start'
]

retention_activity = events_clean[
    events_clean['event_name'].isin(retention_events)
].copy()

retention_activity['event_timestamp'] = pd.to_datetime(
    retention_activity['event_timestamp']
)

retention_activity['event_date'] = (
    retention_activity['event_timestamp'].dt.normalize()
)

print("Total retention activity events:",
      len(retention_activity))

print("\nEvent breakdown:")
print(
    retention_activity['event_name'].value_counts()
)

Total retention activity events: 157664

Event breakdown:
event_name
core_action           73584
session_start         38928
feature_view          36792
subscription_start     8360
Name: count, dtype: int64


In [265]:
d1_activity = retention_users[
    ['user_id', 'signup_date', 'experiment_group']
].copy()

d1_activity['d1_date'] = (
    d1_activity['signup_date'] +
    pd.Timedelta(days=1)
)

d1_activity = d1_activity.merge(
    retention_activity[
        ['user_id', 'event_date']
    ].drop_duplicates(),
    on='user_id',
    how='left'
)

d1_activity['d1_retained'] = (
    d1_activity['event_date'] ==
    d1_activity['d1_date']
)

d1_retention = (
    d1_activity
    .groupby('user_id')['d1_retained']
    .any()
    .reset_index()
)

d1_retention = d1_retention.merge(
    retention_users[
        ['user_id', 'experiment_group']
    ],
    on='user_id',
    how='left'
)

d1_summary = (
    d1_retention
    .groupby('experiment_group')['d1_retained']
    .agg(
        users='count',
        retained_users='sum',
        retention_rate='mean'
    )
    .reset_index()
)

d1_summary['retention_rate'] *= 100

d1_summary.round(2)

,experiment_group,users,retained_users,retention_rate
0,Control,49963,4556,9.12
1,Treatment,49987,5235,10.47


In [266]:
d7_activity = retention_users[
    ['user_id', 'signup_date', 'experiment_group']
].copy()

d7_activity['d7_date'] = (
    d7_activity['signup_date'] +
    pd.Timedelta(days=7)
)

d7_activity = d7_activity.merge(
    retention_activity[
        ['user_id', 'event_date']
    ].drop_duplicates(),
    on='user_id',
    how='left'
)

d7_activity['d7_retained'] = (
    d7_activity['event_date'] ==
    d7_activity['d7_date']
)

d7_retention = (
    d7_activity
    .groupby('user_id')['d7_retained']
    .any()
    .reset_index()
)

d7_retention = d7_retention.merge(
    retention_users[
        ['user_id', 'experiment_group']
    ],
    on='user_id',
    how='left'
)

d7_summary = (
    d7_retention
    .groupby('experiment_group')['d7_retained']
    .agg(
        users='count',
        retained_users='sum',
        retention_rate='mean'
    )
    .reset_index()
)

d7_summary['retention_rate'] *= 100

d7_summary.round(2)

,experiment_group,users,retained_users,retention_rate
0,Control,49963,7873,15.76
1,Treatment,49987,8977,17.96


In [267]:
d30_activity = retention_users[
    ['user_id', 'signup_date', 'experiment_group']
].copy()

d30_activity['d30_date'] = (
    d30_activity['signup_date'] +
    pd.Timedelta(days=30)
)

d30_activity = d30_activity.merge(
    retention_activity[
        ['user_id', 'event_date']
    ].drop_duplicates(),
    on='user_id',
    how='left'
)

d30_activity['d30_retained'] = (
    d30_activity['event_date'] ==
    d30_activity['d30_date']
)

d30_retention = (
    d30_activity
    .groupby('user_id')['d30_retained']
    .any()
    .reset_index()
)

d30_retention = d30_retention.merge(
    retention_users[
        ['user_id', 'experiment_group']
    ],
    on='user_id',
    how='left'
)

d30_summary = (
    d30_retention
    .groupby('experiment_group')['d30_retained']
    .agg(
        users='count',
        retained_users='sum',
        retention_rate='mean'
    )
    .reset_index()
)

d30_summary['retention_rate'] *= 100

d30_summary.round(2)

,experiment_group,users,retained_users,retention_rate
0,Control,49963,557,1.11
1,Treatment,49987,619,1.24


In [268]:
from statsmodels.stats.proportion import proportions_ztest

control_d1 = d1_retention[
    d1_retention['experiment_group'] == 'Control'
]

treatment_d1 = d1_retention[
    d1_retention['experiment_group'] == 'Treatment'
]

count = np.array([
    control_d1['d1_retained'].sum(),
    treatment_d1['d1_retained'].sum()
])

nobs = np.array([
    len(control_d1),
    len(treatment_d1)
])

z_d1, p_d1 = proportions_ztest(
    count,
    nobs
)

print(f"D1 z-statistic: {z_d1:.4f}")
print(f"D1 p-value: {p_d1:.10f}")

D1 z-statistic: -7.2001
D1 p-value: 0.0000000000


In [269]:
control_d7 = d7_retention[
    d7_retention['experiment_group'] == 'Control'
]

treatment_d7 = d7_retention[
    d7_retention['experiment_group'] == 'Treatment'
]

count = np.array([
    control_d7['d7_retained'].sum(),
    treatment_d7['d7_retained'].sum()
])

nobs = np.array([
    len(control_d7),
    len(treatment_d7)
])

z_d7, p_d7 = proportions_ztest(
    count,
    nobs
)

print(f"D7 z-statistic: {z_d7:.4f}")
print(f"D7 p-value: {p_d7:.10f}")

D7 z-statistic: -9.2932
D7 p-value: 0.0000000000


In [270]:
control_d30 = d30_retention[
    d30_retention['experiment_group'] == 'Control'
]

treatment_d30 = d30_retention[
    d30_retention['experiment_group'] == 'Treatment'
]

count = np.array([
    control_d30['d30_retained'].sum(),
    treatment_d30['d30_retained'].sum()
])

nobs = np.array([
    len(control_d30),
    len(treatment_d30)
])

z_d30, p_d30 = proportions_ztest(
    count,
    nobs
)

print(f"D30 z-statistic: {z_d30:.4f}")
print(f"D30 p-value: {p_d30:.10f}")

D30 z-statistic: -1.8104
D30 p-value: 0.0702331004


In [271]:
from statsmodels.stats.proportion import confint_proportions_2indep

control_d1_users = len(control_d1)
treatment_d1_users = len(treatment_d1)

control_d1_retained = control_d1['d1_retained'].sum()
treatment_d1_retained = treatment_d1['d1_retained'].sum()

ci_low_d1, ci_high_d1 = confint_proportions_2indep(
    treatment_d1_retained,
    treatment_d1_users,
    control_d1_retained,
    control_d1_users,
    method='wald'
)

d1_lift = (
    treatment_d1_retained / treatment_d1_users
    -
    control_d1_retained / control_d1_users
)

print(f"D1 observed lift: {d1_lift * 100:.2f} pp")
print(f"D1 95% CI lower: {ci_low_d1 * 100:.2f} pp")
print(f"D1 95% CI upper: {ci_high_d1 * 100:.2f} pp")

D1 observed lift: 1.35 pp
D1 95% CI lower: 0.99 pp
D1 95% CI upper: 1.72 pp


In [272]:
from statsmodels.stats.proportion import confint_proportions_2indep

control_d7_users = len(control_d7)
treatment_d7_users = len(treatment_d7)

control_d7_retained = control_d7['d7_retained'].sum()
treatment_d7_retained = treatment_d7['d7_retained'].sum()

ci_low_d7, ci_high_d7 = confint_proportions_2indep(
    treatment_d7_retained,
    treatment_d7_users,
    control_d7_retained,
    control_d7_users,
    method='wald'
)

d7_lift = (
    treatment_d7_retained / treatment_d7_users
    -
    control_d7_retained / control_d7_users
)

print(f"D7 observed lift: {d7_lift * 100:.2f} pp")
print(f"D7 95% CI lower: {ci_low_d7 * 100:.2f} pp")
print(f"D7 95% CI upper: {ci_high_d7 * 100:.2f} pp")

D7 observed lift: 2.20 pp
D7 95% CI lower: 1.74 pp
D7 95% CI upper: 2.67 pp


In [273]:
from statsmodels.stats.proportion import confint_proportions_2indep

control_d30_users = len(control_d30)
treatment_d30_users = len(treatment_d30)

control_d30_retained = control_d30['d30_retained'].sum()
treatment_d30_retained = treatment_d30['d30_retained'].sum()

ci_low_d30, ci_high_d30 = confint_proportions_2indep(
    treatment_d30_retained,
    treatment_d30_users,
    control_d30_retained,
    control_d30_users,
    method='wald'
)

d30_lift = (
    treatment_d30_retained / treatment_d30_users
    -
    control_d30_retained / control_d30_users
)

print(f"D30 observed lift: {d30_lift * 100:.2f} pp")
print(f"D30 95% CI lower: {ci_low_d30 * 100:.2f} pp")
print(f"D30 95% CI upper: {ci_high_d30 * 100:.2f} pp")

D30 observed lift: 0.12 pp
D30 95% CI lower: -0.01 pp
D30 95% CI upper: 0.26 pp


In [277]:
successful_transactions = transactions_clean[
    transactions_clean['payment_status'].str.lower() == 'success'
]

paid_users = (
    successful_transactions
    .groupby('user_id')
    .size()
    .reset_index(name='successful_transactions')
)

paid_users['paid'] = 1

experiment_revenue = experiment_activation.merge(
    paid_users[['user_id', 'paid']],
    on='user_id',
    how='left'
)

experiment_revenue['paid'] = (
    experiment_revenue['paid']
    .fillna(0)
    .astype(int)
)

paid_conversion = (
    experiment_revenue
    .groupby('experiment_group')
    .agg(
        users=('user_id', 'nunique'),
        paid_users=('paid', 'sum')
    )
)

paid_conversion['paid_conversion_rate'] = (
    paid_conversion['paid_users'] /
    paid_conversion['users'] * 100
)

paid_conversion

,users,paid_users,paid_conversion_rate
experiment_group,,,
Control,49963,3574,7.153293
Treatment,49987,4285,8.572229


In [278]:
control_rate = paid_conversion.loc[
    'Control', 'paid_conversion_rate'
]

treatment_rate = paid_conversion.loc[
    'Treatment', 'paid_conversion_rate'
]

absolute_lift = treatment_rate - control_rate

relative_lift = (
    absolute_lift / control_rate
) * 100

print(f"Control paid conversion: {control_rate:.2f}%")
print(f"Treatment paid conversion: {treatment_rate:.2f}%")
print(f"Absolute lift: +{absolute_lift:.2f} percentage points")
print(f"Relative lift: +{relative_lift:.2f}%")

Control paid conversion: 7.15%
Treatment paid conversion: 8.57%
Absolute lift: +1.42 percentage points
Relative lift: +19.84%


In [279]:
from statsmodels.stats.proportion import proportions_ztest

# Number of paying users
successes = [
    paid_conversion.loc['Control', 'paid_users'],
    paid_conversion.loc['Treatment', 'paid_users']
]

# Total users
sample_sizes = [
    paid_conversion.loc['Control', 'users'],
    paid_conversion.loc['Treatment', 'users']
]

# Two-proportion z-test
z_stat, p_value = proportions_ztest(
    successes,
    sample_sizes
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.10f}")

Z-statistic: -8.3333
P-value: 0.0000000000


In [280]:
import numpy as np
from statsmodels.stats.proportion import confint_proportions_2indep

control_paid = paid_conversion.loc['Control', 'paid_users']
control_users = paid_conversion.loc['Control', 'users']

treatment_paid = paid_conversion.loc['Treatment', 'paid_users']
treatment_users = paid_conversion.loc['Treatment', 'users']

# Confidence interval for Treatment - Control
ci_low, ci_high = confint_proportions_2indep(
    treatment_paid,
    treatment_users,
    control_paid,
    control_users,
    method='wald'
)

print(f"Treatment - Control lift: {(treatment_rate - control_rate):.4f} percentage points")
print(f"95% CI lower: {ci_low * 100:.4f} percentage points")
print(f"95% CI upper: {ci_high * 100:.4f} percentage points")

Treatment - Control lift: 1.4189 percentage points
95% CI lower: 1.0853 percentage points
95% CI upper: 1.7525 percentage points


In [281]:
# Successful transactions only
successful_transactions = transactions_clean[
    transactions_clean['payment_status'].str.lower() == 'success'
].copy()

# Calculate total revenue per user
revenue_by_user = (
    successful_transactions
    .groupby('user_id')['amount']
    .sum()
    .reset_index(name='total_revenue')
)

# Merge revenue with experiment users
experiment_revenue = experiment_revenue.drop(
    columns=['total_revenue'],
    errors='ignore'
)

experiment_revenue = experiment_revenue.merge(
    revenue_by_user,
    on='user_id',
    how='left'
)

# Users with no successful transaction have zero revenue
experiment_revenue['total_revenue'] = (
    experiment_revenue['total_revenue']
    .fillna(0)
)

# Revenue per user by experiment group
revenue_per_user = (
    experiment_revenue
    .groupby('experiment_group')
    .agg(
        users=('user_id', 'nunique'),
        total_revenue=('total_revenue', 'sum')
    )
)

revenue_per_user['revenue_per_user'] = (
    revenue_per_user['total_revenue'] /
    revenue_per_user['users']
)

revenue_per_user

,users,total_revenue,revenue_per_user
experiment_group,,,
Control,49963,2383888.30,47.713074
Treatment,49987,2821453.82,56.443752


In [282]:
control_rpu = revenue_per_user.loc[
    'Control', 'revenue_per_user'
]

treatment_rpu = revenue_per_user.loc[
    'Treatment', 'revenue_per_user'
]

revenue_absolute_lift = treatment_rpu - control_rpu

revenue_relative_lift = (
    revenue_absolute_lift / control_rpu
) * 100

print(f"Control revenue/user: ${control_rpu:.2f}")
print(f"Treatment revenue/user: ${treatment_rpu:.2f}")
print(f"Absolute lift: ${revenue_absolute_lift:.2f} per user")
print(f"Relative lift: {revenue_relative_lift:.2f}%")

Control revenue/user: $47.71
Treatment revenue/user: $56.44
Absolute lift: $8.73 per user
Relative lift: 18.30%


In [283]:
# Calculate revenue per paying user
revenue_per_paying_user = (
    experiment_revenue[
        experiment_revenue['paid'] == 1
    ]
    .groupby('experiment_group')
    .agg(
        paying_users=('user_id', 'nunique'),
        total_revenue=('total_revenue', 'sum')
    )
)

revenue_per_paying_user['revenue_per_paying_user'] = (
    revenue_per_paying_user['total_revenue'] /
    revenue_per_paying_user['paying_users']
)

revenue_per_paying_user

,paying_users,total_revenue,revenue_per_paying_user
experiment_group,,,
Control,3574,2383888.30,667.008478
Treatment,4285,2821453.82,658.448966


In [284]:
control_rppu = revenue_per_paying_user.loc[
    'Control', 'revenue_per_paying_user'
]

treatment_rppu = revenue_per_paying_user.loc[
    'Treatment', 'revenue_per_paying_user'
]

rppu_absolute_lift = treatment_rppu - control_rppu

rppu_relative_lift = (
    rppu_absolute_lift / control_rppu
) * 100

print(f"Control revenue/paying user: ${control_rppu:.2f}")
print(f"Treatment revenue/paying user: ${treatment_rppu:.2f}")
print(f"Absolute lift: ${rppu_absolute_lift:.2f}")
print(f"Relative lift: {rppu_relative_lift:.2f}%")

Control revenue/paying user: $667.01
Treatment revenue/paying user: $658.45
Absolute lift: $-8.56
Relative lift: -1.28%


In [286]:
successful_transactions_with_group = successful_transactions.merge(
    experiment_activation[['user_id', 'experiment_group']],
    on='user_id',
    how='inner'
)

aov_analysis = (
    successful_transactions_with_group
    .groupby('experiment_group')
    .agg(
        successful_transactions=('transaction_id', 'nunique'),
        total_revenue=('amount', 'sum')
    )
)

aov_analysis['aov'] = (
    aov_analysis['total_revenue'] /
    aov_analysis['successful_transactions']
)

aov_analysis

,successful_transactions,total_revenue,aov
experiment_group,,,
Control,3574,2383888.30,667.008478
Treatment,4285,2821453.82,658.448966


In [287]:
control_transactions = aov_analysis.loc[
    'Control', 'successful_transactions'
]

treatment_transactions = aov_analysis.loc[
    'Treatment', 'successful_transactions'
]

transaction_absolute_lift = (
    treatment_transactions - control_transactions
)

transaction_relative_lift = (
    transaction_absolute_lift / control_transactions
) * 100

print(f"Control successful transactions: {control_transactions:,}")
print(f"Treatment successful transactions: {treatment_transactions:,}")
print(f"Additional successful transactions: {transaction_absolute_lift:,}")
print(f"Relative lift: {transaction_relative_lift:.2f}%")

Control successful transactions: 3,574
Treatment successful transactions: 4,285
Additional successful transactions: 711
Relative lift: 19.89%


In [288]:
incremental_revenue_per_user = (
    treatment_rpu - control_rpu
)

print(
    f"Incremental revenue per user: "
    f"${incremental_revenue_per_user:.2f}"
)

Incremental revenue per user: $8.73


In [289]:
rollout_users = 100_000

estimated_incremental_revenue = (
    incremental_revenue_per_user * rollout_users
)

print(f"Rollout population: {rollout_users:,} users")
print(
    f"Estimated incremental revenue: "
    f"${estimated_incremental_revenue:,.2f}"
)

Rollout population: 100,000 users
Estimated incremental revenue: $873,067.81


In [290]:
activation_lift = (
    experiment_activation
    .groupby('experiment_group')['activated']
    .mean()
)

activation_absolute_lift = (
    activation_lift['Treatment'] -
    activation_lift['Control']
)

rollout_users = 100_000

incremental_activated_users = (
    activation_absolute_lift * rollout_users
)

print(
    f"Activation lift: "
    f"{activation_absolute_lift * 100:.2f} pp"
)

print(
    f"Estimated incremental activated users "
    f"for {rollout_users:,} users: "
    f"{incremental_activated_users:,.0f}"
)

Activation lift: 4.93 pp
Estimated incremental activated users for 100,000 users: 4,927


In [294]:

# Count unique users at each funnel stage by experiment group

funnel_counts = (
    funnel
    .groupby('experiment_group')[['user_id', 'event_name']]
    .apply(
        lambda x: pd.Series({
            'Signup': x.loc[
                x['event_name'] == 'signup',
                'user_id'
            ].nunique(),

            'Onboarding Start': x.loc[
                x['event_name'] == 'onboarding_start',
                'user_id'
            ].nunique(),

            'Onboarding Complete': x.loc[
                x['event_name'] == 'onboarding_complete',
                'user_id'
            ].nunique(),

            'Core Action': x.loc[
                x['event_name'] == 'core_action',
                'user_id'
            ].nunique()
        })
    )
)

funnel_counts

,Signup,Onboarding Start,Onboarding Complete,Core Action
experiment_group,,,,
Control,49963,49963,35935,17142
Treatment,49987,49987,38899,19631


In [295]:
funnel_counts = (
    funnel
    .groupby(['experiment_group', 'event_name'])['user_id']
    .nunique()
    .unstack(fill_value=0)
)

# Make sure all funnel stages exist
for stage in funnel_events:
    if stage not in funnel_counts.columns:
        funnel_counts[stage] = 0

# Keep stages in the correct order
funnel_counts = funnel_counts[
    funnel_events
]

# Rename columns for readability
funnel_counts.columns = [
    'Signup',
    'Onboarding Start',
    'Onboarding Complete',
    'Core Action'
]

funnel_counts

,Signup,Onboarding Start,Onboarding Complete,Core Action
experiment_group,,,,
Control,49963,49963,35935,17142
Treatment,49987,49987,38899,19631


In [296]:
# Calculate stage-to-stage conversion rates

stage_conversion = pd.DataFrame(index=funnel_counts.index)

stage_conversion['Signup → Onboarding Start'] = (
    funnel_counts['Onboarding Start'] /
    funnel_counts['Signup'] * 100
)

stage_conversion['Onboarding Start → Complete'] = (
    funnel_counts['Onboarding Complete'] /
    funnel_counts['Onboarding Start'] * 100
)

stage_conversion['Onboarding Complete → Core Action'] = (
    funnel_counts['Core Action'] /
    funnel_counts['Onboarding Complete'] * 100
)

stage_conversion

,Signup → Onboarding Start,Onboarding Start → Complete,Onboarding Complete → Core Action
experiment_group,,,
Control,100.0,71.923223,47.702797
Treatment,100.0,77.818233,50.466593


In [297]:
# Users who viewed at least one feature
feature_users = (
    events_clean[
        events_clean['event_name'] == 'feature_view'
    ]
    [['user_id']]
    .drop_duplicates()
)

feature_users['feature_adopted'] = 1

# Add experiment group
feature_analysis = experiment_activation.merge(
    feature_users,
    on='user_id',
    how='left'
)

feature_analysis['feature_adopted'] = (
    feature_analysis['feature_adopted']
    .fillna(0)
    .astype(int)
)

# Feature adoption rate
feature_adoption = (
    feature_analysis
    .groupby('experiment_group')
    .agg(
        users=('user_id', 'nunique'),
        feature_users=('feature_adopted', 'sum')
    )
)

feature_adoption['feature_adoption_rate'] = (
    feature_adoption['feature_users'] /
    feature_adoption['users'] * 100
)

feature_adoption

,users,feature_users,feature_adoption_rate
experiment_group,,,
Control,49963,17142,34.309389
Treatment,49987,19631,39.272211


In [298]:
# Calculate unique sessions per user

sessions_per_user = (
    events_clean
    .groupby('user_id')['session_id']
    .nunique()
    .reset_index(name='sessions')
)

# Attach experiment group
session_analysis = experiment_activation[
    ['user_id', 'experiment_group']
].merge(
    sessions_per_user,
    on='user_id',
    how='left'
)

# Users with no recorded sessions
session_analysis['sessions'] = (
    session_analysis['sessions']
    .fillna(0)
)

# Average sessions per user
session_summary = (
    session_analysis
    .groupby('experiment_group')
    .agg(
        users=('user_id', 'nunique'),
        total_sessions=('sessions', 'sum'),
        avg_sessions_per_user=('sessions', 'mean')
    )
)

session_summary

,users,total_sessions,avg_sessions_per_user
experiment_group,,,
Control,49963,49963,1.0
Treatment,49987,49987,1.0


In [299]:
# Define meaningful engagement events
engagement_events = [
    'core_action',
    'feature_view',
    'subscription_start'
]

# Keep meaningful events
engagement = events_clean[
    events_clean['event_name'].isin(engagement_events)
].copy()

# Count meaningful actions per user
engagement_per_user = (
    engagement
    .groupby('user_id')
    .size()
    .reset_index(name='meaningful_actions')
)

# Attach experiment group
engagement_analysis = experiment_activation[
    ['user_id', 'experiment_group']
].merge(
    engagement_per_user,
    on='user_id',
    how='left'
)

# Users with no meaningful actions
engagement_analysis['meaningful_actions'] = (
    engagement_analysis['meaningful_actions']
    .fillna(0)
)

# Summary
engagement_summary = (
    engagement_analysis
    .groupby('experiment_group')
    .agg(
        users=('user_id', 'nunique'),
        total_actions=('meaningful_actions', 'sum'),
        avg_actions_per_user=('meaningful_actions', 'mean')
    )
)

engagement_summary

,users,total_actions,avg_actions_per_user
experiment_group,,,
Control,49963,55220.0,1.105218
Treatment,49987,63456.0,1.269450


In [300]:
control_actions = engagement_summary.loc[
    'Control', 'avg_actions_per_user'
]

treatment_actions = engagement_summary.loc[
    'Treatment', 'avg_actions_per_user'
]

action_absolute_lift = (
    treatment_actions - control_actions
)

action_relative_lift = (
    action_absolute_lift / control_actions
) * 100

print(
    f"Control actions/user: "
    f"{control_actions:.4f}"
)

print(
    f"Treatment actions/user: "
    f"{treatment_actions:.4f}"
)

print(
    f"Absolute lift: "
    f"+{action_absolute_lift:.4f} actions/user"
)

print(
    f"Relative lift: "
    f"+{action_relative_lift:.2f}%"
)

Control actions/user: 1.1052
Treatment actions/user: 1.2695
Absolute lift: +0.1642 actions/user
Relative lift: +14.86%


In [301]:
from scipy.stats import ttest_ind

control_actions_data = engagement_analysis.loc[
    engagement_analysis['experiment_group'] == 'Control',
    'meaningful_actions'
]

treatment_actions_data = engagement_analysis.loc[
    engagement_analysis['experiment_group'] == 'Treatment',
    'meaningful_actions'
]

t_stat, p_value = ttest_ind(
    control_actions_data,
    treatment_actions_data,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.10f}")

T-statistic: -16.4853
P-value: 0.0000000000


In [302]:
root_cause_summary = pd.DataFrame({
    'Metric': [
        'Onboarding Completion',
        'Core Action',
        '7-Day Activation',
        'Feature Adoption',
        'Meaningful Actions/User',
        'Paid Conversion',
        'D1 Retention',
        'D7 Retention',
        'D30 Retention',
        'Sessions/User'
    ],
    
    'Control': [
        71.92,
        34.31,
        33.44,
        34.31,
        1.1052,
        7.15,
        9.12,
        15.76,
        1.11,
        1.00
    ],
    
    'Treatment': [
        77.82,
        39.27,
        38.37,
        39.27,
        1.2695,
        8.57,
        10.47,
        17.96,
        1.24,
        1.00
    ],
    
    'Direction': [
        'Positive',
        'Positive',
        'Positive',
        'Positive',
        'Positive',
        'Positive',
        'Positive',
        'Positive',
        'No clear evidence',
        'No difference'
    ]
})

root_cause_summary

,Metric,Control,Treatment,Direction
0,Onboarding Completion,71.9200,77.8200,Positive
1,Core Action,34.3100,39.2700,Positive
2,7-Day Activation,33.4400,38.3700,Positive
3,Feature Adoption,34.3100,39.2700,Positive
4,Meaningful Actions/User,1.1052,1.2695,Positive
5,Paid Conversion,7.1500,8.5700,Positive
6,D1 Retention,9.1200,10.4700,Positive
7,D7 Retention,15.7600,17.9600,Positive
8,D30 Retention,1.1100,1.2400,No clear evidence
9,Sessions/User,1.0000,1.0000,No difference


In [303]:
decision_framework = pd.DataFrame({
    'Area': [
        'Primary KPI',
        'Early Engagement',
        'Early Retention',
        'Paid Conversion',
        'Revenue/User',
        'Long-Term Retention',
        'Experiment Validity',
        'Business Risk'
    ],
    
    'Finding': [
        'Activation increased by 4.93 pp',
        'Meaningful actions/user increased by 14.86%',
        'D1 and D7 retention increased significantly',
        'Paid conversion increased by 1.42 pp',
        'Revenue/user increased by 18.30%',
        'D30 improvement was not statistically significant',
        'No major group imbalance; timing sensitivity was robust',
        'Monitor long-term retention and revenue quality'
    ],
    
    'Decision': [
        'Strong positive',
        'Positive',
        'Positive',
        'Strong positive',
        'Strong positive',
        'Needs monitoring',
        'Validated',
        'Manage during rollout'
    ]
})

decision_framework

,Area,Finding,Decision
0,Primary KPI,Activation increased by 4.93 pp,Strong positive
1,Early Engagement,Meaningful actions/user increased by 14.86%,Positive
2,Early Retention,D1 and D7 retention increased significantly,Positive
3,Paid Conversion,Paid conversion increased by 1.42 pp,Strong positive
4,Revenue/User,Revenue/user increased by 18.30%,Strong positive
5,Long-Term Retention,D30 improvement was not statistically significant,Needs monitoring
6,Experiment Validity,No major group imbalance; timing sensitivity w...,Validated
7,Business Risk,Monitor long-term retention and revenue quality,Manage during rollout


In [304]:
executive_summary = pd.DataFrame({
    'KPI': [
        'Experiment Users',
        'Control Activation Rate',
        'Treatment Activation Rate',
        'Activation Lift',
        'Paid Conversion Lift',
        'Revenue/User Lift',
        'D1 Retention Lift',
        'D7 Retention Lift',
        'D30 Retention Lift',
        'Meaningful Actions/User Lift',
        'Estimated Incremental Activated Users',
        'Estimated Incremental Revenue'
    ],

    'Result': [
        '99,950',
        '33.44%',
        '38.37%',
        '+4.93 pp',
        '+1.42 pp',
        '+18.30%',
        '+1.35 pp',
        '+2.20 pp',
        '+0.12 pp (Not Significant)',
        '+14.86%',
        '≈4,930 per 100K users',
        '≈$873K per 100K users'
    ],

    'Business_Interpretation': [
        'Valid users included in the experiment',
        'Baseline activation under old onboarding',
        'Activation under redesigned onboarding',
        'Strong improvement in primary KPI',
        'More users converted to paid',
        'Higher monetization per user',
        'Significant improvement in early retention',
        'Significant improvement in 7-day retention',
        'No statistically significant long-term improvement',
        'Higher meaningful product engagement',
        'Potential additional activated users at scale',
        'Potential incremental revenue at scale'
    ]
})

executive_summary

,KPI,Result,Business_Interpretation
0,Experiment Users,"99,950",Valid users included in the experiment
1,Control Activation Rate,33.44%,Baseline activation under old onboarding
2,Treatment Activation Rate,38.37%,Activation under redesigned onboarding
3,Activation Lift,+4.93 pp,Strong improvement in primary KPI
4,Paid Conversion Lift,+1.42 pp,More users converted to paid
5,Revenue/User Lift,+18.30%,Higher monetization per user
6,D1 Retention Lift,+1.35 pp,Significant improvement in early retention
7,D7 Retention Lift,+2.20 pp,Significant improvement in 7-day retention
8,D30 Retention Lift,+0.12 pp (Not Significant),No statistically significant long-term improve...
9,Meaningful Actions/User Lift,+14.86%,Higher meaningful product engagement


In [305]:
print("experiment_activation:")
print(experiment_activation.shape)
print(experiment_activation.columns.tolist())

print("\nexperiment_revenue:")
print(experiment_revenue.shape)
print(experiment_revenue.columns.tolist())

print("\nengagement_analysis:")
print(engagement_analysis.shape)
print(engagement_analysis.columns.tolist())

experiment_activation:
(99950, 10)
['user_id', 'signup_date', 'country', 'device_type', 'age_group', 'acquisition_channel', 'experiment_group', 'variant_version', 'assignment_date', 'activated']

experiment_revenue:
(99950, 12)
['user_id', 'signup_date', 'country', 'device_type', 'age_group', 'acquisition_channel', 'experiment_group', 'variant_version', 'assignment_date', 'activated', 'paid', 'total_revenue']

engagement_analysis:
(99950, 3)
['user_id', 'experiment_group', 'meaningful_actions']


In [306]:
# ==========================================
# STEP 16.2 — POWER BI MASTER DATASET
# ==========================================

# Start with the validated experiment + revenue table
powerbi_users = experiment_revenue.copy()

# Rename columns for clarity
powerbi_users = powerbi_users.rename(
    columns={
        'activated': 'activated_7d',
        'total_revenue': 'revenue'
    }
)

# --------------------------------------------------
# 1. Add meaningful actions
# --------------------------------------------------

powerbi_users = powerbi_users.merge(
    engagement_analysis[
        ['user_id', 'meaningful_actions']
    ],
    on='user_id',
    how='left'
)

powerbi_users['meaningful_actions'] = (
    powerbi_users['meaningful_actions']
    .fillna(0)
)

# --------------------------------------------------
# 2. Add sessions
# --------------------------------------------------

sessions_per_user = (
    events_clean
    .groupby('user_id')['session_id']
    .nunique()
    .reset_index(name='sessions')
)

powerbi_users = powerbi_users.merge(
    sessions_per_user,
    on='user_id',
    how='left'
)

powerbi_users['sessions'] = (
    powerbi_users['sessions']
    .fillna(0)
)

# --------------------------------------------------
# 3. Add feature adoption
# --------------------------------------------------

feature_users = (
    events_clean[
        events_clean['event_name'] == 'feature_view'
    ][['user_id']]
    .drop_duplicates()
)

feature_users['feature_adopted'] = 1

powerbi_users = powerbi_users.merge(
    feature_users,
    on='user_id',
    how='left'
)

powerbi_users['feature_adopted'] = (
    powerbi_users['feature_adopted']
    .fillna(0)
    .astype(int)
)

# --------------------------------------------------
# 4. Final column order
# --------------------------------------------------

powerbi_users = powerbi_users[
    [
        'user_id',
        'signup_date',
        'country',
        'device_type',
        'age_group',
        'acquisition_channel',
        'experiment_group',
        'variant_version',
        'assignment_date',
        'activated_7d',
        'feature_adopted',
        'meaningful_actions',
        'sessions',
        'paid',
        'revenue'
    ]
]

# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

print("Power BI dataset shape:", powerbi_users.shape)

print("\nColumns:")
print(powerbi_users.columns.tolist())

print("\nMissing values:")
print(powerbi_users.isna().sum())

print("\nDuplicate users:")
print(powerbi_users['user_id'].duplicated().sum())

print("\nExperiment groups:")
print(powerbi_users['experiment_group'].value_counts())

powerbi_users.head()

Power BI dataset shape: (99950, 15)

Columns:
['user_id', 'signup_date', 'country', 'device_type', 'age_group', 'acquisition_channel', 'experiment_group', 'variant_version', 'assignment_date', 'activated_7d', 'feature_adopted', 'meaningful_actions', 'sessions', 'paid', 'revenue']

Missing values:
user_id                0
signup_date            0
country                0
device_type            0
age_group              0
acquisition_channel    0
experiment_group       0
variant_version        0
assignment_date        0
activated_7d           0
feature_adopted        0
meaningful_actions     0
sessions               0
paid                   0
revenue                0
dtype: int64

Duplicate users:
0

Experiment groups:
experiment_group
Treatment    49987
Control      49963
Name: count, dtype: int64


,user_id,signup_date,country,device_type,age_group,acquisition_channel,experiment_group,variant_version,assignment_date,activated_7d,feature_adopted,meaningful_actions,sessions,paid,revenue
0,U000001,2026-02-21,USA,Desktop,45-54,Paid Search,Treatment,v2,2026-02-21,1,1,4.0,1,1,361.56
1,U000002,2026-01-15,Canada,Mobile,55+,Paid Search,Treatment,v2,2026-01-15,0,0,0.0,1,0,0.00
2,U000003,2026-03-13,India,Mobile,25-34,Organic,Treatment,v2,2026-03-13,0,0,0.0,1,0,0.00
3,U000004,2026-03-02,USA,Mobile,45-54,Social,Control,v1,2026-03-02,1,1,3.0,1,0,0.00
4,U000005,2026-01-21,UK,Desktop,18-24,Organic,Control,v1,2026-01-21,0,0,0.0,1,0,0.00


In [308]:
# STEP 16.2 — Create Power BI Master Dataset

# Start with the experiment + revenue dataset
powerbi_master = experiment_revenue.copy()

# Rename columns for Power BI
powerbi_master = powerbi_master.rename(columns={
    'activated': 'activated_7d',
    'total_revenue': 'revenue'
})

# ---------------------------------------------------------
# 1. Add meaningful actions
# ---------------------------------------------------------

powerbi_master = powerbi_master.merge(
    engagement_analysis[['user_id', 'meaningful_actions']],
    on='user_id',
    how='left'
)

# ---------------------------------------------------------
# 2. Calculate sessions per user
# ---------------------------------------------------------

session_counts = (
    events_clean.groupby('user_id')['session_id']
    .nunique()
    .reset_index()
    .rename(columns={'session_id': 'sessions'})
)

powerbi_master = powerbi_master.merge(
    session_counts,
    on='user_id',
    how='left'
)

# ---------------------------------------------------------
# 3. Calculate feature adoption
# ---------------------------------------------------------

feature_users = (
    events_clean.loc[
        events_clean['event_name'] == 'feature_view',
        'user_id'
    ]
    .drop_duplicates()
)

powerbi_master['feature_adopted'] = (
    powerbi_master['user_id']
    .isin(feature_users)
    .astype(int)
)

# ---------------------------------------------------------
# 4. Handle missing values
# ---------------------------------------------------------

powerbi_master['meaningful_actions'] = (
    powerbi_master['meaningful_actions']
    .fillna(0)
)

powerbi_master['sessions'] = (
    powerbi_master['sessions']
    .fillna(0)
)

powerbi_master['revenue'] = (
    powerbi_master['revenue']
    .fillna(0)
)

powerbi_master['paid'] = (
    powerbi_master['paid']
    .fillna(0)
    .astype(int)
)

powerbi_master['activated_7d'] = (
    powerbi_master['activated_7d']
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 5. Arrange final columns
# ---------------------------------------------------------

powerbi_master = powerbi_master[
    [
        'user_id',
        'signup_date',
        'country',
        'device_type',
        'age_group',
        'acquisition_channel',
        'experiment_group',
        'variant_version',
        'assignment_date',
        'activated_7d',
        'feature_adopted',
        'meaningful_actions',
        'sessions',
        'paid',
        'revenue'
    ]
]

# ---------------------------------------------------------
# 6. Display preview
# ---------------------------------------------------------

print("Power BI Master Dataset created successfully!")
print("\nShape:", powerbi_master.shape)

display(powerbi_master.head())

Power BI Master Dataset created successfully!

Shape: (99950, 15)


,user_id,signup_date,country,device_type,age_group,acquisition_channel,experiment_group,variant_version,assignment_date,activated_7d,feature_adopted,meaningful_actions,sessions,paid,revenue
0,U000001,2026-02-21,USA,Desktop,45-54,Paid Search,Treatment,v2,2026-02-21,1,1,4.0,1,1,361.56
1,U000002,2026-01-15,Canada,Mobile,55+,Paid Search,Treatment,v2,2026-01-15,0,0,0.0,1,0,0.00
2,U000003,2026-03-13,India,Mobile,25-34,Organic,Treatment,v2,2026-03-13,0,0,0.0,1,0,0.00
3,U000004,2026-03-02,USA,Mobile,45-54,Social,Control,v1,2026-03-02,1,1,3.0,1,0,0.00
4,U000005,2026-01-21,UK,Desktop,18-24,Organic,Control,v1,2026-01-21,0,0,0.0,1,0,0.00


In [309]:
print("Shape:", powerbi_master.shape)

print("\nDuplicate user IDs:",
      powerbi_master['user_id'].duplicated().sum())

print("\nMissing values:")
print(powerbi_master.isnull().sum())

print("\nExperiment groups:")
print(powerbi_master['experiment_group'].value_counts())

print("\nData types:")
print(powerbi_master.dtypes)

Shape: (99950, 15)

Duplicate user IDs: 0

Missing values:
user_id                0
signup_date            0
country                0
device_type            0
age_group              0
acquisition_channel    0
experiment_group       0
variant_version        0
assignment_date        0
activated_7d           0
feature_adopted        0
meaningful_actions     0
sessions               0
paid                   0
revenue                0
dtype: int64

Experiment groups:
experiment_group
Treatment    49987
Control      49963
Name: count, dtype: int64

Data types:
user_id                        object
signup_date            datetime64[ns]
country                        object
device_type                    object
age_group                      object
acquisition_channel            object
experiment_group               object
variant_version                object
assignment_date                object
activated_7d                    int32
feature_adopted                 int32
meaningful_actions   

In [310]:
powerbi_master['assignment_date'] = pd.to_datetime(
    powerbi_master['assignment_date']
)

print(powerbi_master[['signup_date', 'assignment_date']].dtypes)

signup_date        datetime64[ns]
assignment_date    datetime64[ns]
dtype: object


In [311]:
# STEP 16.3 — Create Power BI Retention Table

# Meaningful activity events used for retention
retention_events = [
    'core_action',
    'feature_view',
    'session_start',
    'subscription_start'
]

# Keep only meaningful retention events
retention_activity = events_clean[
    events_clean['event_name'].isin(retention_events)
].copy()

# Make sure timestamps are datetime
retention_activity['event_timestamp'] = pd.to_datetime(
    retention_activity['event_timestamp']
)

# Create event date
retention_activity['event_date'] = (
    retention_activity['event_timestamp'].dt.normalize()
)

# Create signup date from master dataset
signup_lookup = powerbi_master[
    ['user_id', 'signup_date']
].copy()

signup_lookup['signup_date'] = pd.to_datetime(
    signup_lookup['signup_date']
).dt.normalize()

# Add signup date to events
retention_activity = retention_activity.merge(
    signup_lookup,
    on='user_id',
    how='inner'
)

# Calculate days since signup
retention_activity['days_since_signup'] = (
    retention_activity['event_date']
    - retention_activity['signup_date']
).dt.days

# Users retained on D1
d1_users = set(
    retention_activity.loc[
        retention_activity['days_since_signup'] == 1,
        'user_id'
    ]
)

# Users retained on D7
d7_users = set(
    retention_activity.loc[
        retention_activity['days_since_signup'] == 7,
        'user_id'
    ]
)

# Users retained on D30
d30_users = set(
    retention_activity.loc[
        retention_activity['days_since_signup'] == 30,
        'user_id'
    ]
)

# Create retention table
retention_table = powerbi_master[
    ['user_id', 'experiment_group']
].copy()

retention_table['d1_retained'] = (
    retention_table['user_id']
    .isin(d1_users)
    .astype(int)
)

retention_table['d7_retained'] = (
    retention_table['user_id']
    .isin(d7_users)
    .astype(int)
)

retention_table['d30_retained'] = (
    retention_table['user_id']
    .isin(d30_users)
    .astype(int)
)

# Preview
print("Retention table created successfully!")
print("\nShape:", retention_table.shape)

display(retention_table.head())

Retention table created successfully!

Shape: (99950, 5)


,user_id,experiment_group,d1_retained,d7_retained,d30_retained
0,U000001,Treatment,0,1,0
1,U000002,Treatment,0,0,0
2,U000003,Treatment,0,0,0
3,U000004,Control,0,0,0
4,U000005,Control,0,0,0


In [312]:
# STEP 16.4 — Export Power BI-ready datasets

import os

output_folder = "/mnt/data/novatech_powerbi_data"

os.makedirs(output_folder, exist_ok=True)

# Export master dataset
powerbi_master.to_csv(
    os.path.join(output_folder, "powerbi_master.csv"),
    index=False
)

# Export retention table
retention_table.to_csv(
    os.path.join(output_folder, "retention_table.csv"),
    index=False
)

print("Power BI files exported successfully!")
print("\nFiles created:")

for file in os.listdir(output_folder):
    print("-", file)

Power BI files exported successfully!

Files created:
- powerbi_master.csv
- retention_table.csv


In [315]:
import os

# Windows Downloads folder
downloads_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads"
)

# Save the two Power BI files there
powerbi_master.to_csv(
    os.path.join(downloads_folder, "powerbi_master.csv"),
    index=False
)

retention_table.to_csv(
    os.path.join(downloads_folder, "retention_table.csv"),
    index=False
)

print("Files saved successfully!")
print("\nLocation:")
print(downloads_folder)

print("\nFiles:")
print(os.path.join(downloads_folder, "powerbi_master.csv"))
print(os.path.join(downloads_folder, "retention_table.csv"))

Files saved successfully!

Location:
C:\Users\patel\Downloads

Files:
C:\Users\patel\Downloads\powerbi_master.csv
C:\Users\patel\Downloads\retention_table.csv
